<a href="https://colab.research.google.com/github/ohjeonguk/Project2-TinyGPTmodel/blob/main/notebook_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — MLP Character Model on `names.txt`

이제 `names.txt`는 그대로 두고, **dataset을 일반화**하고 **모델을 MLP로 확장**합니다.

- bigram: 현재 문자 1개만 사용
- MLP: 길이 `block_size`의 context 전체 사용
- 표현도 one-hot 대신 **embedding**을 사용

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
chars = ['.'] + chars
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
encoded_words = [[stoi[ch] for ch in w] for w in words]

## 1. 일반화된 context dataset

In [ ]:
class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X, self.Y = [], []
        for word in encoded_words:
            context = [0] * block_size
            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)
                context = context[1:] + [ix]
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

block_size = 3
dataset = NamesContextDataset(encoded_words, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 3])
yb.shape: torch.Size([64])


## 2. MLP model

In [ ]:
class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=10, hidden_dim=200):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([64, 27])
initial loss: 3.4355385303497314


## 3. Train / Eval

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

## 4. 학습

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.3573
epoch  5 | train loss 2.2745
epoch 10 | train loss 2.2688
epoch 15 | train loss 2.2668
epoch 20 | train loss 2.2655
epoch 25 | train loss 2.2640
epoch 29 | train loss 2.2642


## 5. Sampling

In [ ]:
@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval()
    results = []
    for _ in range(num_samples):
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = []
        for _ in range(max_len):
            logits = model(context)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()
            if next_token == 0:
                break
            out.append(itos[next_token])
            context = torch.cat([context[:, 1:], ix], dim=1)
        results.append("".join(out))
    return results

sample(model, block_size, itos, device, num_samples=10)

['shce',
 'marva',
 'ashidmynn',
 'kan',
 'roy',
 'hertia',
 'kamiah',
 'ericketi',
 'kaidonalviah',
 'johe']

## 6. 정리

- dataset은 `context -> next char` 구조를 유지합니다.
- 차이는 모델 내부입니다.
- embedding과 MLP를 사용하므로 bigram보다 더 풍부한 문맥을 처리할 수 있습니다.